# Supplementary Tables S7 and S9: Cross-validation and forecasting validation of the multinomial logit model

This notebook evaluates Model (6) of Supplementary Table S6 with 5-fold stratified cross-validation (Table S7) and with a forecasting validation that trains on 1990-2016 and tests on 2017 (Table S9). Performance is McFadden's pseudo-$R^2$ on the held-out data.

It reads the processed data in the data folder (`path2data`) and can be run on its own.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns

from vizutils import pvalue2stars, table_row, make_multinomial_latex_table, logistic_coefficient_plot

import pyscisci.all as pyscisci

# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')

from sklearn.model_selection import StratifiedKFold

Please install python-Levenshtein to take full advantage of the NLP tools. 
 pip install python-Levenshtein
Please install sparse_dot_topn to take full advantage of the NLP tools. 
 pip install sparse_dot_topn


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


# The citation preference network

In [2]:
# lets focus on the citation without the same organization and self citations

country_auc_df = pd.read_csv(os.path.join(path2data, 'oa_countrycites_nosameorg_auc.csv.gz'))

country_auc_df = country_auc_df[country_auc_df['N']>=50]

country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5)/np.sqrt(country_auc_df['Cov'])
# one-sided test in the direction of the observed preference: H1 is AUC > 0.5 (over-citation)
# when Z > 0, and AUC < 0.5 (under-citation) when Z < 0
country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

def reject_holm(pvalues):
    reject, pvalues, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
    return reject
country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

net_auc = country_auc_df[country_auc_df['CitingCountry'] != country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
net_auc = net_auc[net_auc['Significant']]
self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
self_auc = self_auc[self_auc['Significant']]

syear, eyear = 1990, 2017
net_auc = net_auc[net_auc['CitedYear'] >= syear]
net_auc = net_auc[net_auc['CitedYear'] <= eyear]

comm2country = {0:['AE', 'BD', 'CN', 'CY', 'DZ', 'EG', 'ID', 'IN', 'IQ', 'IR', 'JO', 'KH', 'KR', 'KW', 'MA', 'MY', 'OM', 'PK', 'QA', 'SA', 'SG', 'TH', 'TN', 'TR', 'TW', 'VN'],
1:['AR', 'BN', 'BO', 'BR', 'BW', 'BY', 'CL', 'CM', 'CO', 'CR', 'GE', 'HK', 'HR', 'LU', 'MZ', 'NC', 'PA', 'PE', 'RU', 'SJ', 'ST', 'UY', 'VE'],
2:['AT', 'BE', 'CA', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HU', 'IE', 'IL', 'IS', 'IT', 'JP', 'MX', 'NL', 'NO', 'NP', 'NZ', 'PH', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK'],
3:['AU', 'CH', 'GB', 'US'],
4:['BH', 'ET', 'GH', 'KE', 'MW', 'NG', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']}

country2comm = {c:ci for ci,clist in comm2country.items() for c in clist}
poscountrylist = sorted(list(country2comm.keys()))

connected_countries = np.sort(np.unique(net_auc[net_auc['Zscore'] > 0][['CitingCountry', 'CitedCountry']].values.flatten()))

# The country-pair panel of Model (6)

In [3]:
panel_data = pd.read_csv(os.path.join(path2data, 'link_prediction_panel_data_1990_2017.csv.gz'))

net_panel = panel_data.merge(net_auc.rename(columns={'CitingCountry':'oCountry', 'CitedCountry':'tCountry','CitedYear':'year'}), 
                           how='left', on=['oCountry','tCountry','year'])

net_panel['edge_preference'] = 1 + (net_panel['Zscore'] < 0).astype(int)
net_panel.loc[net_panel['Significant'].isnull(), 'edge_preference'] = 0

# the citation preferences cover 1990-2017, so the panel rows of later years are not used
net_panel = net_panel[(net_panel['year'] >= syear) & (net_panel['year'] <= eyear)].reset_index(drop=True)

v1=['oGDP_PCAP','tGDP_PCAP', 'oPop','tPop']
v2=['PhysicalDistance','SameContinent']
v3=['comlang_off','TopicSimilarity']
v4=['cumulative_laws']
v5=['CollabStrength']
v6=['oTopFrac','tTopFrac']

# the variables of Model (6) in Supplementary Table S6
model_features = v1 + v2 + v3 + v4 + v5 + v6
dep_var = 'edge_preference'

net_panel = net_panel.dropna(subset=model_features).reset_index(drop=True)
print(net_panel.shape)


def design_matrices(train, test):
    """Standardize the continuous variables with the training mean and standard deviation, and add the year fixed effects."""
    X_train = train[model_features + ['year']].copy()
    X_test = test[model_features + ['year']].copy()
    for c in model_features:
        if X_train[c].nunique() > 2:
            mean, std = X_train[c].mean(), X_train[c].std()
            X_train[c] = (X_train[c] - mean)/std
            X_test[c] = (X_test[c] - mean)/std
    X_train = pd.get_dummies(X_train, columns=['year'], drop_first=True, dtype=int)
    X_test = pd.get_dummies(X_test, columns=['year'], drop_first=True, dtype=int).reindex(columns=X_train.columns, fill_value=0)
    return sm.add_constant(X_train, has_constant='add'), sm.add_constant(X_test, has_constant='add')


def heldout_pseudo_r2(result, X_test, y_test, y_train):
    """McFadden's pseudo R2 on held-out data; the null model predicts the class shares of the training set."""
    eps = 1e-15
    y_test = np.asarray(y_test).astype(int)
    pred = np.asarray(result.predict(X_test))
    loglike = np.log(pred[np.arange(len(y_test)), y_test] + eps).sum()
    class_shares = np.bincount(np.asarray(y_train).astype(int), minlength=pred.shape[1])/len(y_train)
    loglike_null = np.log(class_shares[y_test] + eps).sum()
    return 1 - loglike/loglike_null

(744760, 26)


## Supplementary Table S7: 5-fold stratified cross-validation

In [4]:
y = net_panel[dep_var].astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

kfold = []
for fold, (train_index, test_index) in enumerate(skf.split(net_panel, y)):
    train, test = net_panel.iloc[train_index], net_panel.iloc[test_index]
    X_train, X_test = design_matrices(train, test)
    result = sm.MNLogit(y.iloc[train_index].values, X_train.values).fit(disp=0, maxiter=200)
    kfold.append([fold + 1, len(train), len(test), result.prsquared,
                  heldout_pseudo_r2(result, X_test.values, y.iloc[test_index], y.iloc[train_index])])

kfold = pd.DataFrame(kfold, columns=['Fold', 'Training observations', 'Test observations', 'Pseudo R2 (training)',
                                     'Pseudo R2 (test)'])
print(kfold.round(4).to_string(index=False))
print('Mean {:.4f}, standard deviation {:.4f}'.format(kfold['Pseudo R2 (test)'].mean(), kfold['Pseudo R2 (test)'].std()))

 Fold  Training observations  Test observations  Pseudo R2 (training)  Pseudo R2 (test)
    1                 595808             148952                0.6394            0.6348
    2                 595808             148952                0.6411            0.6279
    3                 595808             148952                0.6401            0.6321
    4                 595808             148952                0.6372            0.6436
    5                 595808             148952                0.6360            0.6488
Mean 0.6374, standard deviation 0.0086


## Supplementary Table S9: forecasting validation (train 1990-2016, test 2017)

In [5]:
train = net_panel[net_panel['year'] <= 2016]
test = net_panel[net_panel['year'] == 2017]
X_train, X_test = design_matrices(train, test)
# 2017 has no year fixed effect in the training data, so the test year uses the 2016 fixed effect
X_test['year_2016'] = 1

result = sm.MNLogit(train[dep_var].astype(int).values, X_train.values).fit(disp=0, maxiter=200)

forecast = pd.DataFrame({'Dataset':['Training set (1990-2016)', 'Test set (2017)'],
                         'Observations':[len(train), len(test)],
                         'Pseudo R2':[result.prsquared, heldout_pseudo_r2(result, X_test.values, test[dep_var], train[dep_var])]})
print(forecast.round(4).to_string(index=False))

                 Dataset  Observations  Pseudo R2
Training set (1990-2016)        715690     0.6393
         Test set (2017)         29070     0.6284


## Source Data for Supplementary Tables 7 and 9

The values reported in the tables, written to `Source Data/SourceData_7_TableS7.xlsx` and `SourceData_9_TableS9.xlsx`.

In [6]:
# the Source Data helpers (sourcedatautils.py, in the code folder like vizutils.py)
from sourcedatautils import write_source_data

model_description = ('Model (6) of Supplementary Table 6: multinomial logit with temporal fixed-effects predicting a positive '
                     'or negative citation preference compared to no preference, 1990-2017 country-pair panel. Continuous '
                     'variables are standardized with the mean and standard deviation of the training set. Performance is '
                     "McFadden's pseudo R2; on the test set, the null model predicts the class shares of the training set.")

summary = pd.DataFrame({'Statistic':['Mean', 'Standard deviation'],
                        'Pseudo R2 (test)':[kfold['Pseudo R2 (test)'].mean(), kfold['Pseudo R2 (test)'].std()]})
write_source_data('SourceData_7_TableS7.xlsx',
                  'Supplementary Table 7: Performance of the 5-fold stratified cross-validation for Model (6).',
                  model_description + ' Folds are stratified by the citation preference (random_state = 100).',
                  [('Folds', 'Pseudo R2 of each fold.', kfold), ('Summary', 'Mean and standard deviation over the folds.', summary)])

write_source_data('SourceData_9_TableS9.xlsx',
                  'Supplementary Table 9: Performance of the forecasting validation for Model (6).',
                  model_description + ' The model is trained on 1990-2016 and tested on 2017, which uses the 2016 year fixed effect.',
                  [('Forecasting', 'Pseudo R2 of the training and test sets.', forecast)])

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_7_TableS7.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_9_TableS9.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
